In [3]:
# Cell 1: Setup and Imports for Semantic Segmentation
import os
import glob
os.system('pip uninstall numpy -Y')
os.system('pip install numpy-1.26.4')
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.utils import to_categorical
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("Initialized script for Semantic Segmentation model.")

# Define image dimensions (should match output from preprocessing)
IMG_HEIGHT = 512 # Thermal image height
IMG_WIDTH = 640  # Thermal image width
IMG_CHANNELS = 3 # RGB image channels

# Define segmentation classes for Fuel Load & Vegetation Stress Mapping
# IMPORTANT: These are example classes. You will need to define your actual classes
# and ensure your ground truth masks are labeled accordingly.
# Example: 0=Background/Non-vegetated, 1=Healthy Trees, 2=Stressed Trees, 3=Dry Grass, 4=Dense Brush, 5=Water, 6=Urban/Road
NUM_CLASSES = 7 

# Training parameters
BATCH_SIZE = 8
EPOCHS = 50 # This is a placeholder, actual epochs will depend on dataset size and convergence

# Path to your preprocessed data from the previous step
OUTPUT_BASE_DIR = 'processed_flame3_dataset'

logging.info(f"Image dimensions set to: {IMG_HEIGHT}x{IMG_WIDTH}x{IMG_CHANNELS}")
logging.info(f"Number of segmentation classes: {NUM_CLASSES}")
logging.info(f"Batch size: {BATCH_SIZE}, Epochs: {EPOCHS}")
logging.info(f"Loading preprocessed data from: {OUTPUT_BASE_DIR}")

print("Setup complete. Ready to load and prepare data.")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# Cell 2: Data Loading and Preprocessing for Segmentation

def load_image_and_mask(image_path, mask_path, img_height, img_width, num_classes):
    """
    Loads an RGB image and its corresponding segmentation mask,
    resizes them, and performs necessary preprocessing (normalization, one-hot encoding).
    """
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        logging.error(f"Failed to load image: {image_path}")
        return None, None
    image = cv2.resize(image, (img_width, img_height))
    image = image / 255.0 # Normalize to 0-1 range

    # Load mask
    # IMPORTANT: For semantic segmentation, you NEED pixel-level ground truth masks.
    # These masks should have integer values corresponding to your NUM_CLASSES (e.g., 0, 1, 2...).
    # The FLAME 3 dataset might provide fire/no-fire masks, but for Fuel Load/Vegetation Stress,
    # you would need specifically labeled masks for those categories.
    # This is a placeholder for loading such a mask.
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) # Assuming grayscale mask where pixel values are class IDs
    if mask is None:
        logging.error(f"Failed to load mask: {mask_path}. Ensure masks exist and are correctly named.")
        return None, None
    mask = cv2.resize(mask, (img_width, img_height), interpolation=cv2.INTER_NEAREST) # Use INTER_NEAREST for masks to preserve class IDs

    # One-hot encode the mask
    # Reshape mask to (height, width, 1) before one-hot encoding if it's 2D
    if mask.ndim == 2:
        mask = np.expand_dims(mask, axis=-1)
    mask = to_categorical(mask, num_classes=num_classes)
    
    return image, mask

def data_generator(image_paths, mask_paths, img_height, img_width, num_classes, batch_size):
    """
    A generator that yields batches of images and masks for training.
    """
    num_samples = len(image_paths)
    indices = np.arange(num_samples)
    
    while True:
        np.random.shuffle(indices) # Shuffle data for each epoch
        for i in range(0, num_samples, batch_size):
            batch_indices = indices[i:i+batch_size]
            batch_images = []
            batch_masks = []
            
            for idx in batch_indices:
                img, msk = load_image_and_mask(image_paths[idx], mask_paths[idx], img_height, img_width, num_classes)
                if img is not None and msk is not None:
                    batch_images.append(img)
                    batch_masks.append(msk)
                else:
                    logging.warning(f"Skipping sample {image_paths[idx]} due to loading error.")

            if batch_images and batch_masks: # Only yield if batch is not empty
                yield np.array(batch_images), np.array(batch_masks)

# Example usage (assuming your preprocessed data is in OUTPUT_BASE_DIR)
# IMPORTANT: You need to have ground truth segmentation masks for training.
# For this example, we'll assume a dummy mask path structure.
# In a real scenario, mask_paths would point to your actual labeled masks.

# Discover preprocessed aligned RGB images
fire_rgb_files = glob.glob(os.path.join(OUTPUT_BASE_DIR, 'Fire', '*_aligned_rgb.jpg'))
no_fire_rgb_files = glob.glob(os.path.join(OUTPUT_BASE_DIR, 'No_Fire', '*_aligned_rgb.jpg'))
all_rgb_images = sorted(fire_rgb_files + no_fire_rgb_files)

# Placeholder for mask paths - YOU MUST REPLACE THIS WITH YOUR ACTUAL MASK PATHS
# Assuming masks are in a 'masks' subfolder corresponding to the image structure
# e.g., 'processed_flame3_dataset/Fire/masks/00001_mask.png'
all_mask_paths = []
for img_path in all_rgb_images:
    # This is a hypothetical mask path based on image path. Adjust as per your mask storage.
    # Example: 'processed_flame3_dataset/Fire/00001_aligned_rgb.jpg' -> 'path/to/ground_truth_masks/Fire/00001_mask.png'
    # You will need to map your RGB image to its corresponding segmentation mask.
    # For FLAME 3, if masks are provided, they might be in a separate directory or named differently.
    
    # Let's assume for this example, a mask named '00001_mask.png' exists for '00001_aligned_rgb.jpg'
    # in a parallel 'masks' directory structure.
    base_name = os.path.basename(img_path).replace('_aligned_rgb.jpg', '')
    category_dir = os.path.basename(os.path.dirname(img_path))
    
    # THIS IS A CRITICAL ASSUMPTION. ADJUST TO YOUR ACTUAL MASK LOCATION.
    # For example, if your masks are in a 'ground_truth_masks' folder at the same level as 'processed_flame3_dataset'
    # and follow a similar category/filename structure:
    # mask_root = 'path/to/your/ground_truth_masks'
    # mask_path = os.path.join(mask_root, category_dir, f"{base_name}_mask.png") # Or .tiff, .jpg etc.
    
    # For now, let's use a dummy path that will likely fail unless you create it.
    dummy_mask_path = img_path.replace('_aligned_rgb.jpg', '_mask.png').replace(OUTPUT_BASE_DIR, 'path/to/your/segmentation_masks')
    all_mask_paths.append(dummy_mask_path)


# Verify that mask paths are being generated (will likely be non-existent initially)
logging.info(f"Discovered {len(all_rgb_images)} aligned RGB images.")
logging.info(f"Generated {len(all_mask_paths)} hypothetical mask paths. (Verify these paths point to actual masks!)")

# Create data generators
# train_generator = data_generator(train_image_paths, train_mask_paths, ...)
# val_generator = data_generator(val_image_paths, val_mask_paths, ...)

print("Data loading and preprocessing functions defined. Remember to provide actual ground truth masks.")

In [ ]:
# Cell 3: U-Net Model Definition for Semantic Segmentation

def conv_block(input_tensor, num_filters):
    """A convolutional block with two Conv2D layers, BatchNormalization, and ReLU activation."""
    x = Conv2D(num_filters, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Conv2D(num_filters, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(x)
    x = BatchNormalization()(x)
    return x

def build_unet(input_shape, num_classes):
    """
    Builds a U-Net model for semantic segmentation.
    Input:
        input_shape (tuple): Shape of the input images (height, width, channels).
        num_classes (int): Number of segmentation classes.
    Output:
        A Keras Model object representing the U-Net.
    """
    inputs = Input(input_shape)

    # Encoder (Contracting Path)
    # Block 1
    conv1 = conv_block(inputs, 64)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)
    pool1 = Dropout(0.1)(pool1)

    # Block 2
    conv2 = conv_block(pool1, 128)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)
    pool2 = Dropout(0.1)(pool2)

    # Block 3
    conv3 = conv_block(pool2, 256)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)
    pool3 = Dropout(0.2)(pool3)

    # Block 4
    conv4 = conv_block(pool3, 512)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)
    pool4 = Dropout(0.2)(pool4)

    # Bottleneck
    conv5 = conv_block(pool4, 1024)
    conv5 = Dropout(0.3)(conv5)

    # Decoder (Expanding Path)
    # Block 6
    up6 = Conv2D(512, (2, 2), activation='relu', padding='same', kernel_initializer='he_normal')(UpSampling2D(size=(2, 2))(conv5))
    merge6 = concatenate([conv4, up6], axis=3) # Skip connection
    merge6 = Dropout(0.2)(merge6)
    conv6 = conv_block(merge6, 512)

    # Block 7
    up7 = Conv2D(256, (2, 2), activation='relu', padding='same', kernel_initializer='he_normal')(UpSampling2D(size=(2, 2))(conv6))
    merge7 = concatenate([conv3, up7], axis=3) # Skip connection
    merge7 = Dropout(0.2)(merge7)
    conv7 = conv_block(merge7, 256)

    # Block 8
    up8 = Conv2D(128, (2, 2), activation='relu', padding='same', kernel_initializer='he_normal')(UpSampling2D(size=(2, 2))(conv7))
    merge8 = concatenate([conv2, up8], axis=3) # Skip connection
    merge8 = Dropout(0.1)(merge8)
    conv8 = conv_block(merge8, 128)

    # Block 9
    up9 = Conv2D(64, (2, 2), activation='relu', padding='same', kernel_initializer='he_normal')(UpSampling2D(size=(2, 2))(conv8))
    merge9 = concatenate([conv1, up9], axis=3) # Skip connection
    merge9 = Dropout(0.1)(merge9)
    conv9 = conv_block(merge9, 64)

    # Output layer
    outputs = Conv2D(num_classes, (1, 1), activation='softmax')(conv9)

    model = Model(inputs=inputs, outputs=outputs)
    return model

# Build the model
input_shape = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
unet_model = build_unet(input_shape, NUM_CLASSES)

logging.info("U-Net model architecture defined.")
unet_model.summary(line_length=120) # Print model summary for review

print("U-Net model built. Ready for compilation and training.")

In [ ]:
# Cell 4: Model Compilation and Training (Placeholder for actual training)

# Compile the U-Net model
unet_model.compile(optimizer=Adam(learning_rate=1e-4), 
                   loss=categorical_crossentropy, 
                   metrics=['accuracy'])

logging.info("U-Net model compiled.")

# --- Placeholder for Training Data Preparation ---
# In a real scenario, you would split your all_rgb_images and all_mask_paths
# into training and validation sets.
# For demonstration, let's assume a simple split.
# This assumes you have actual mask files at the paths generated in Cell 2.

# IMPORTANT: Replace 'path/to/your/segmentation_masks' with the actual root of your masks
# and ensure the mask naming convention matches your image naming convention.
# Example: if image is 'processed_flame3_dataset/Fire/00001_aligned_rgb.jpg',
# its mask might be 'path/to/your/segmentation_masks/Fire/00001_mask.png'
# You need to create this mapping correctly.

# For a basic example, let's create a dummy mask path for each image path
# This is for code structure demonstration and will likely fail without real masks.
# In a real scenario, you would have actual mask files.
# For now, we'll use the dummy paths generated in Cell 2.

if not all_rgb_images:
    logging.error("No aligned RGB images found. Cannot proceed with training.")
    print("No images found for training. Please ensure Cell 2 ran correctly and your INPUT_BASE_DIR is set to a valid dataset path.")
else:
    # Simple split for demonstration (e.g., 80% train, 20% validation)
    split_ratio = 0.8
    split_index = int(len(all_rgb_images) * split_ratio)

    train_image_paths = all_rgb_images[:split_index]
    train_mask_paths = all_mask_paths[:split_index] # Assuming masks are aligned with images
    
    val_image_paths = all_rgb_images[split_index:]
    val_mask_paths = all_mask_paths[split_index:] # Assuming masks are aligned with images

    if not train_image_paths or not val_image_paths:
        logging.error("Not enough images to create proper train/validation sets. Ensure your dataset is populated.")
        print("Not enough images to create train/validation sets. Please check your dataset.")
    else:
        logging.info(f"Training on {len(train_image_paths)} samples, validating on {len(val_image_paths)} samples.")

        train_gen = data_generator(train_image_paths, train_mask_paths, IMG_HEIGHT, IMG_WIDTH, NUM_CLASSES, BATCH_SIZE)
        val_gen = data_generator(val_image_paths, val_mask_paths, IMG_HEIGHT, IMG_WIDTH, NUM_CLASSES, BATCH_SIZE)

        steps_per_epoch_train = len(train_image_paths) // BATCH_SIZE
        steps_per_epoch_val = len(val_image_paths) // BATCH_SIZE

        if steps_per_epoch_train == 0 or steps_per_epoch_val == 0:
            logging.error("Steps per epoch is zero. Batch size might be too large or dataset too small.")
            print("Steps per epoch is zero. Adjust BATCH_SIZE or check dataset size.")
        else:
            logging.info("Starting U-Net model training...")
            # Train the model
            # history = unet_model.fit(
            #     train_gen,
            #     steps_per_epoch=steps_per_epoch_train,
            #     epochs=EPOCHS,
            #     validation_data=val_gen,
            #     validation_steps=steps_per_epoch_val
            # )
            logging.warning("Model training is commented out. Uncomment 'unet_model.fit(...)' to start training.")
            print("Model compilation complete. Training code is commented out. Uncomment `unet_model.fit(...)` to begin training.")

            # You might want to save the trained model after training
            # unet_model.save('unet_fuel_segmentation_model.h5')
            # logging.info("U-Net model saved to 'unet_fuel_segmentation_model.h5'")

In [ ]:
# Cell 5: Prediction and Visualization (Sample Output)

import matplotlib.pyplot as plt

# Define a color map for visualization of segmentation classes
# IMPORTANT: Adjust these colors and ensure they correspond to your NUM_CLASSES
# and the class IDs in your ground truth masks.
CLASS_COLORS = {
    0: [0, 0, 0],       # Black for Background/Non-vegetated
    1: [0, 128, 0],     # Dark Green for Healthy Trees
    2: [128, 128, 0],   # Olive for Stressed Trees
    3: [210, 105, 30],  # Chocolate for Dry Grass
    4: [139, 69, 19],   # Saddle Brown for Dense Brush
    5: [0, 0, 255],     # Blue for Water
    6: [128, 128, 128]  # Gray for Urban/Road
}

def apply_color_map(mask, class_colors):
    """Applies a color map to a grayscale class ID mask."""
    colored_mask = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
    for class_id, color in class_colors.items():
        colored_mask[mask == class_id] = color
    return colored_mask

def predict_and_visualize(model, image_path, img_height, img_width, num_classes, class_colors):
    """
    Loads an image, performs segmentation prediction, and visualizes the result.
    """
    logging.info(f"Loading image for prediction: {image_path}")
    image = cv2.imread(image_path)
    if image is None:
        logging.error(f"Failed to load image for prediction: {image_path}")
        return

    original_display_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # For display

    # Preprocess image for model input
    image = cv2.resize(image, (img_width, img_height))
    image = image / 255.0
    image = np.expand_dims(image, axis=0) # Add batch dimension

    logging.info("Performing prediction...")
    prediction = model.predict(image)[0] # Get prediction for the single image
    
    # Convert prediction (probabilities) to class IDs
    predicted_mask = np.argmax(prediction, axis=-1)

    # Apply color map for visualization
    colored_predicted_mask = apply_color_map(predicted_mask, class_colors)

    logging.info("Displaying results.")
    plt.figure(figsize=(18, 8))

    plt.subplot(1, 2, 1)
    plt.imshow(original_display_image)
    plt.title('Original Aligned RGB Image')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(colored_predicted_mask)
    plt.title('Predicted Segmentation (Fuel/Vegetation Stress)')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# --- Example Usage ---
# IMPORTANT: For this to work, you need a trained model.
# If you uncommented and ran Cell 4's training, the model is 'unet_model'.
# Otherwise, you would need to load a pre-trained model:
# from tensorflow.keras.models import load_model
# unet_model = load_model('unet_fuel_segmentation_model.h5') # If you saved it

if 'unet_model' in locals() and unet_model is not None:
    # Find a sample image from your preprocessed data
    sample_image_files = glob.glob(os.path.join(OUTPUT_BASE_DIR, 'Fire', '*_aligned_rgb.jpg'))
    if sample_image_files:
        sample_image_path = sample_image_files[0]
        print(f"\n--- Displaying Sample Segmentation Prediction for: {os.path.basename(sample_image_path)} ---")
        predict_and_visualize(unet_model, sample_image_path, IMG_HEIGHT, IMG_WIDTH, NUM_CLASSES, CLASS_COLORS)
    else:
        print(f"No sample aligned RGB images found in '{os.path.join(OUTPUT_BASE_DIR, 'Fire')}'. Cannot perform prediction.")
else:
    print("U-Net model not found or not trained. Please ensure Cell 4 ran successfully or load a pre-trained model.")
    print("Prediction cannot be performed without a trained model.")

print("\n--- Sample Prediction and Visualization Setup Complete ---")